In [0]:
from pyspark.sql.functions import col, sha2

In [0]:
SOURCE_CATALOG_NAME = 'covid_19'
SOURCE_SCHEMA_NAME = 'silver'
SOURCE_TABLE_NAME1 = 'locations'
SOURCE_TABLE_NAME2 = 'vaccinations'

TARGET_CATALOG_NAME = 'covid_19'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'dim_country'

In [0]:
df_country_from_locations = (
    spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME1}')
    .select('country', 'iso_code')
)

In [0]:
df_country_from_vaccinations = (
    spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME2}')
    .select('country', 'iso_code')
)

In [0]:
df_dim_country = (
    df_country_from_locations
    .unionByName(df_country_from_vaccinations)
    .dropDuplicates(['iso_code'])
    .withColumn(
        'country_key',
        sha2(col('iso_code'), 256)
    )
    .select(
        'country_key',
        'country',
        'iso_code'
    )
)

In [0]:
df_dim_country\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')